# 04 -- Pareto analysis of acoustic conspicuousness vs surface area

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antnewman/acoustic-beacon-optimiser/blob/main/notebooks/04_pareto_analysis.ipynb)

Compute the Pareto frontier of integrated conspicuousness (IC) vs
surface area (SA) over the spherical-cap family using NSGA-II, then
overlay the natural reflector geometries from the literature.

**Headline question:** are the natural geometries Pareto-optimal?
If yes, that is evidence for acoustic optimality under natural
selection. If the natural points sit significantly inside the
frontier, there are additional constraints (structural,
developmental, phylogenetic) not captured in the acoustic model.

**Runtime:** 1-3 hours on a single CPU for the default grid; scale
the population and generation counts up for production runs.

**Setup in Colab:** run `!pip install -q git+https://github.com/antnewman/acoustic-beacon-optimiser.git` in the first cell, then continue.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from abo.acoustics.objectives import (
    integrated_conspicuousness,
    surface_area,
)
from abo.acoustics.target_strength import monostatic_target_strength
from abo.biology.call_spectra import glossophaga_call_spectrum
from abo.biology.natural_shapes import MARCGRAVIA_EVENIA, MUCUNA_HOLTONII
from abo.geometry.meshing import mesh_spherical_cap
from abo.geometry.reflectors import SphericalCap
from abo.optimisation.encoding import SphericalCapEncoding
from abo.optimisation.multi_objective import pareto_frontier
from abo.optimisation.runners import EvaluationConfig, make_pareto_objectives

## Problem setup

In [ ]:
POP_SIZE = 20
N_GEN = 20
SEED = 42

frequencies = np.linspace(45_000.0, 100_000.0, 8)
angles = np.linspace(0.0, np.pi / 3, 7)
spectrum = glossophaga_call_spectrum(frequencies)
config = EvaluationConfig(
    frequencies=frequencies,
    angles=angles,
    call_spectrum=spectrum,
)

## Run NSGA-II

In [ ]:
encoder = SphericalCapEncoding()
objectives = make_pareto_objectives(encoder, config)

result = pareto_frontier(
    objectives=objectives,
    bounds=encoder.default_bounds,
    population_size=POP_SIZE,
    n_generations=N_GEN,
    seed=SEED,
)

ic_values = -result.pareto_front[:, 0]
sa_values = result.pareto_front[:, 1]

print(f"Pareto points: {len(ic_values)}")
print(f"Evaluations:   {result.n_evals}")
print(f"IC range:      [{ic_values.min():.2f}, {ic_values.max():.2f}] dB")
print(f"SA range:      [{sa_values.min()*1e6:.1f}, {sa_values.max()*1e6:.1f}] mm^2")

## Evaluate natural reflectors at the same BEM configuration

In [ ]:
def cap_from_diameter_depth(diameter_mm, depth_mm):
    a = diameter_mm / 2.0 * 1e-3
    d = depth_mm * 1e-3
    r = (a**2 + d**2) / (2.0 * d)
    theta = float(np.arcsin(a / r))
    return SphericalCap(radius=r, half_angle=theta)

def evaluate_cap(cap):
    grid = mesh_spherical_cap(cap, frequency=float(frequencies.max()))
    ts = monostatic_target_strength(
        grid, frequencies, angles, far_field_range=1.0,
    )
    ic = integrated_conspicuousness(ts, frequencies, angles, spectrum)
    sa = surface_area(grid)
    return ic, sa

natural_points = []
for nat in (MARCGRAVIA_EVENIA, MUCUNA_HOLTONII):
    cap = cap_from_diameter_depth(nat.diameter_mm, nat.depth_mm)
    ic, sa = evaluate_cap(cap)
    natural_points.append((nat.species, ic, sa))
    print(f"{nat.species}: IC={ic:.2f} dB, SA={sa*1e6:.1f} mm^2")

## Plot the Pareto frontier with natural geometries overlaid

In [ ]:
order = np.argsort(sa_values)
ic_sorted = ic_values[order]
sa_sorted = sa_values[order]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sa_sorted * 1e6, ic_sorted, "o-", linewidth=1.5, label="Pareto frontier")
for name, ic, sa in natural_points:
    ax.scatter([sa * 1e6], [ic], s=80, label=name)
ax.set_xlabel("Surface area (mm^2)")
ax.set_ylabel("Integrated conspicuousness (dB)")
ax.set_title("IC vs SA Pareto frontier: spherical cap family vs natural geometries")
ax.grid(visible=True, linestyle=":", linewidth=0.5)
ax.legend()
fig.tight_layout()
fig.savefig("../paper/figures/fig04_pareto_frontier.png", dpi=200)
plt.show()

## Save data for the paper

Results are written to `../results/` which is git-excluded. The paper
will cite a Zenodo DOI for the final data release.

In [ ]:
import os
os.makedirs("../results", exist_ok=True)
np.savez(
    "../results/pareto_spherical_cap.npz",
    pareto_set=result.pareto_set,
    pareto_front=result.pareto_front,
    ic_values=ic_values,
    sa_values=sa_values,
    n_evals=result.n_evals,
    frequencies=frequencies,
    angles=angles,
    natural_points=np.array(
        [(name, ic, sa) for name, ic, sa in natural_points],
        dtype=[("name", "U32"), ("ic", "f8"), ("sa", "f8")],
    ),
)
print("Saved to ../results/pareto_spherical_cap.npz")